# Exploração textual

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/correa-ufrrj/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_04/02_exploracao_textual.ipynb)

O Notebook 01 descreveu a composição e as medidas documentais da base. Agora
voltaremos ao conteúdo linguístico dos registros, começando pelas regras que
transformam texto preservado em unidades analisáveis.

Esta versão usa um **corpus textual sintético menos homogêneo**. O gerador combina
perfis temáticos, vocabulário compartilhado e expressões recorrentes com
probabilidades desiguais. Assim, algumas palavras são muito frequentes, outras são
raras, os temas se sobrepõem e certas colocações reaparecem sem terem contagens
idênticas. Os dados continuam inteiramente fictícios e não constituem evidência
histórica.

## Tokenização e normalização

![O texto preservado passa por normalização, tokenização, filtro e contagem; o diagrama indica possíveis perdas em cada transformação.](imagens/02_fluxo_tokenizacao.svg)

Tokenizar segmenta por regra. Minúsculas, pontuação e stopwords podem apagar
distinções; preserve o texto e documente decisões. Neste exemplo, os tokens são
calculados separadamente por documento para não criar contextos ou n-gramas entre
o fim de um texto e o início do seguinte.


In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_04'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = []
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

In [ ]:
# @title Gerador do corpus textual — perfis lexicais menos homogêneos
# O corpus desta aula é separado do dataset quantitativo da unidade.
import random
import re
from pathlib import Path

import pandas as pd

SEED = 20260925
random.seed(SEED)

# As quatro categorias principais continuam sendo os temas dos documentos.
# Política e cultura funcionam como vocabulários transversais: podem aparecer
# em documentos de qualquer tema e tornam as fronteiras menos artificiais.
COMPOSICAO_TEMATICA = {
    "educação": {"educação": .45, "cultura": .20, "trabalho": .15,
                 "política": .10, "saúde": .05, "progresso": .05},
    "trabalho": {"trabalho": .40, "progresso": .25, "política": .15,
                 "cultura": .10, "educação": .05, "saúde": .05},
    "progresso": {"progresso": .35, "trabalho": .20, "política": .15,
                  "educação": .10, "cultura": .10, "saúde": .10},
    "saúde": {"saúde": .40, "educação": .20, "política": .15,
              "cultura": .10, "progresso": .10, "trabalho": .05},
}

# Cada categoria contém sentenças com pesos diferentes. Os pesos criam uma
# distribuição lexical assimétrica; as sentenças mantêm contexto linguístico,
# ao contrário de uma sequência aleatória de palavras.
REPERTORIOS = {
    "educação": [
        ("A instrução pública amplia o acesso à escola e ao ensino.", 5),
        ("Professores discutem métodos de ensino e formação escolar.", 3),
        ("A escola noturna recebe trabalhadores interessados em aprender.", 3),
        ("O debate menciona leitura, escrita e educação popular.", 2),
        ("Famílias pedem novas escolas para crianças do bairro.", 2),
        ("Bibliotecas e livros são apresentados como meios de instrução.", 1),
    ],
    "trabalho": [
        ("A jornada de trabalho ocupa o centro das reclamações operárias.", 5),
        ("Trabalhadores discutem salário, jornada e condições nas oficinas.", 4),
        ("A associação operária convoca reunião sobre salário e trabalho.", 3),
        ("O conflito nas fábricas envolve patrões, empregados e horários.", 2),
        ("Oficinas contratam novos trabalhadores para diferentes tarefas.", 2),
        ("O emprego irregular afeta famílias de trabalhadores.", 1),
    ],
    "progresso": [
        ("A estrada de ferro é apresentada como sinal de progresso.", 5),
        ("Máquinas e novas oficinas transformam a circulação de mercadorias.", 4),
        ("O progresso material é associado à expansão das estradas.", 3),
        ("A iluminação pública modifica ruas e atividades noturnas.", 2),
        ("Obras urbanas prometem melhorar transportes e comércio.", 2),
        ("A expansão ferroviária também provoca conflitos locais.", 1),
    ],
    "saúde": [
        ("A saúde pública depende de água potável e saneamento.", 5),
        ("Médicos alertam para higiene, água e condições das moradias.", 4),
        ("A campanha de vacinação mobiliza autoridades e moradores.", 3),
        ("Doenças infecciosas preocupam bairros densamente povoados.", 2),
        ("O abastecimento de água aparece como problema de saúde.", 2),
        ("Hospitais registram aumento de atendimentos durante o verão.", 1),
    ],
    "política": [
        ("O governo apresenta novas medidas ao debate público.", 4),
        ("Representantes discutem leis, direitos e responsabilidades.", 3),
        ("A autoridade local recebe críticas de diferentes grupos.", 2),
        ("O debate sobre liberdade e direitos divide opiniões.", 1),
    ],
    "cultura": [
        ("Livros e jornais circulam entre leitores da cidade.", 4),
        ("A biblioteca recebe obras de literatura e história.", 3),
        ("Teatro, música e poesia aparecem no debate cultural.", 2),
        ("Escritores comentam mudanças nos costumes da cidade.", 1),
    ],
}

ABERTURAS = {
    "editorial": [
        "Há questões públicas que exigem atenção continuada.",
        "O debate recente merece consideração mais cuidadosa.",
        "Poucos assuntos revelam de modo tão claro os desafios da vida pública.",
    ],
    "notícia": [
        "Uma reunião realizada nesta semana retomou um assunto já conhecido.",
        "O tema voltou às páginas do jornal após novas manifestações.",
        "Moradores e representantes locais acompanharam uma nova discussão.",
    ],
    "carta": [
        "Senhor redator, escrevo a respeito de um problema que merece atenção.",
        "Peço espaço para comentar uma questão discutida por muitos leitores.",
        "Envio estas linhas para registrar uma preocupação recorrente.",
    ],
}

MARCAS_GENERO = {
    "editorial": [
        "É necessário examinar com cuidado as consequências dessa questão.",
        "A administração deveria apresentar medidas mais consistentes.",
        "Não basta anunciar reformas sem esclarecer seus resultados.",
    ],
    "notícia": [
        "Segundo informações reunidas pela redação, a reunião ocorreu ontem.",
        "Representantes ouvidos pelo jornal relataram posições diferentes.",
        "A decisão deverá ser discutida novamente nos próximos dias.",
    ],
    "carta": [
        "Escrevo ao jornal para chamar atenção para este problema.",
        "Como leitor, peço que as autoridades considerem esta situação.",
        "Espero que estas observações contribuam para o debate público.",
    ],
}

GERAIS = [
    "O assunto voltou a ser discutido nesta semana.",
    "A medida recebeu apoio de parte dos moradores.",
    "Outros participantes apresentaram críticas à proposta.",
    "O debate continuará nas próximas reuniões.",
    "Testemunhos divergem sobre os efeitos da medida.",
    "O jornal registra opiniões favoráveis e contrárias.",
]

DOCUMENTOS = [
    ("D001",1890,"editorial","Capital","educação"),
    ("D002",1891,"notícia","Capital","trabalho"),
    ("D003",1892,"carta","Capital","progresso"),
    ("D004",1893,"editorial","Interior","saúde"),
    ("D005",1894,"notícia","Interior","educação"),
    ("D006",1895,"carta","Interior","trabalho"),
    ("D007",1896,"editorial","Capital","progresso"),
    ("D008",1897,"notícia","Capital","saúde"),
    ("D009",1898,"carta","Capital","educação"),
    ("D010",1899,"editorial","Interior","trabalho"),
    ("D011",1900,"notícia","Interior","progresso"),
    ("D012",1901,"carta","Interior","saúde"),
    ("D013",1890,"editorial","Capital","educação"),
    ("D014",1891,"notícia","Capital","trabalho"),
    ("D015",1892,"carta","Capital","progresso"),
    ("D016",1893,"editorial","Interior","saúde"),
    ("D017",1894,"notícia","Interior","educação"),
    ("D018",1895,"carta","Interior","trabalho"),
    ("D019",1896,"editorial","Capital","progresso"),
    ("D020",1897,"notícia","Capital","saúde"),
    ("D021",1898,"carta","Capital","educação"),
    ("D022",1899,"editorial","Interior","trabalho"),
    ("D023",1900,"notícia","Interior","progresso"),
    ("D024",1901,"carta","Interior","saúde"),
]

def escolha_ponderada(opcoes):
    textos = [texto for texto, _ in opcoes]
    pesos = [peso for _, peso in opcoes]
    return random.choices(textos, weights=pesos, k=1)[0]

def escolher_categoria(tema):
    composicao = COMPOSICAO_TEMATICA[tema]
    return random.choices(
        list(composicao),
        weights=list(composicao.values()),
        k=1,
    )[0]

def gerar_texto(tema, genero, alvo_palavras):
    partes = [random.choice(ABERTURAS[genero])]
    # A marca de gênero não aparece obrigatoriamente em toda sentença.
    contador = len(re.findall(r"[A-Za-zÀ-ÿ]+", partes[0]))

    while contador < alvo_palavras:
        u = random.random()
        if u < .12:
            frase = random.choice(MARCAS_GENERO[genero])
        elif u < .20:
            frase = random.choice(GERAIS)
        else:
            categoria = escolher_categoria(tema)
            frase = escolha_ponderada(REPERTORIOS[categoria])

        partes.append(frase)
        contador += len(re.findall(r"[A-Za-zÀ-ÿ]+", frase))

    return " ".join(partes)

linhas = []
for i, (id_doc, ano, genero, local, tema) in enumerate(DOCUMENTOS):
    # Variação ampla de extensão; D023 permanece como caso deliberadamente longo.
    alvo = random.randint(260, 900)
    if id_doc == "D023":
        alvo = 1700

    texto = gerar_texto(tema, genero, alvo)
    palavras = len(re.findall(r"[A-Za-zÀ-ÿ]+", texto))
    pessoas = max(0, round(random.gauss(palavras / 110, 2)))
    paginas = max(1, round(palavras / random.randint(330, 430)))

    linhas.append({
        "id_documento": id_doc,
        "ano": ano,
        "genero": genero,
        "local": local,
        "tema": tema,
        "palavras": palavras,
        "pessoas": pessoas,
        "paginas": paginas,
        "texto": texto,
    })

dados_gerados = pd.DataFrame(linhas)
Path("dados").mkdir(exist_ok=True)
dados_gerados.to_csv("dados/documentos_textuais.csv", index=False)

print("Dataset textual gerado:", dados_gerados.shape)
print("Extensão mínima, mediana e máxima:",
      dados_gerados["palavras"].min(),
      dados_gerados["palavras"].median(),
      dados_gerados["palavras"].max())
dados_gerados.head(3)

In [ ]:
import math
import re
from collections import Counter

import pandas as pd
import sys
from pathlib import Path
from IPython.display import display
sys.path.insert(0, str(Path.cwd())) if str(Path.cwd()) not in sys.path else None
from graficos import ttr_duplo

dados = pd.read_csv("dados/documentos_textuais.csv")

def tokenizar(texto):
    return re.findall(r"[a-záàâãéêíóôõúç]+", texto.lower())

dados["tokens"] = dados["texto"].map(tokenizar)
todos_tokens = [token for documento in dados["tokens"] for token in documento]
dados[["id_documento", "tokens"]].head(2)

A tokenização transforma cada documento em unidades contáveis. Como as regras
de segmentação e normalização definem o que será reconhecido como token, elas
precisam ser declaradas antes de qualquer frequência.

## Frequências absoluta e relativa

Se $t_i$ é o token na posição $i$, a frequência de uma palavra $w$ é:

$$
f(w)=\sum_{i=1}^{N}\mathbf{1}(t_i=w),
\qquad
p(w)=\frac{f(w)}{N}.
$$

O valor de $p(w)$ só é interpretável quando $N$ está declarado. Neste
experimento mostraremos dois denominadores:

| Medida | Denominador |
|---|---|
| frequência relativa entre todos os tokens | $N$, antes de retirar stopwords |
| frequência relativa entre tokens de conteúdo | $N_c$, depois de retirar stopwords |

Remover stopwords apenas da tabela, mas manter $N$ como denominador, responde a
uma pergunta diferente de recalcular a proporção dentro do vocabulário filtrado.

In [ ]:
stopwords = {"a", "o", "e", "de", "do", "da", "como", "em", "nas", "um", "uma", "também"}
frequencias = Counter(todos_tokens)
tokens_conteudo = [token for token in todos_tokens if token not in stopwords]
frequencias_conteudo = Counter(tokens_conteudo)
total_tokens = len(todos_tokens)
total_tokens_conteudo = len(tokens_conteudo)

tabela_frequencias = pd.DataFrame([
    {
        "palavra": palavra,
        "frequencia": frequencia,
        "relativa_todos_tokens": frequencia / total_tokens,
        "relativa_tokens_conteudo": frequencia / total_tokens_conteudo,
    }
    for palavra, frequencia in frequencias_conteudo.most_common(12)
])
tabela_frequencias

As frequências mostram quanto um termo aparece, mas retiram as ocorrências de
seus contextos. Para avaliar o sentido e a relevância de uma contagem, precisamos
retornar aos documentos que a produziu.

## Concordâncias

Uma concordância recupera uma janela de $j$ tokens à esquerda e à direita de
cada ocorrência. Não é necessário transformar essa operação em uma medida única:
seu papel é devolver contexto ao agregado. As janelas devem respeitar as fronteiras
dos documentos e conservar o identificador da fonte.

In [ ]:
def concordancias(tabela, alvo, janela=4):
    resultados = []
    for _, documento in tabela.iterrows():
        tokens = documento["tokens"]
        for posicao, token in enumerate(tokens):
            if token == alvo:
                inicio = max(0, posicao - janela)
                fim = min(len(tokens), posicao + janela + 1)
                resultados.append({
                    "id_documento": documento["id_documento"],
                    "tema": documento["tema"],
                    "genero": documento["genero"],
                    "contexto": " ".join(tokens[inicio:fim]),
                })
    return pd.DataFrame(resultados)

concordancias(dados, "trabalho", janela=4).head(8)

As concordâncias permitem examinar manualmente o entorno de uma palavra. O
passo seguinte sistematiza uma pergunta relacionada: **quais palavras aparecem
juntas mais vezes do que esperaríamos apenas porque cada uma delas é frequente?**

## N-gramas e colocações

![A frequência do bigrama é comparada às frequências marginais; o resultado deve ser examinado com frequência mínima e concordâncias.](imagens/02_anatomia_pmi.svg)

Um **bigrama** é um par de tokens adjacentes, $(t_i,t_{i+1})$, dentro do mesmo
documento. Sua frequência informa quantas vezes o par foi observado, mas isso não
basta para dizer se há uma associação especialmente característica entre as duas
palavras.

Considere dois casos. Se `de` e `trabalho` aparecem muitas vezes no corpus,
`de trabalho` pode ser frequente simplesmente porque `de` é muito frequente. Já
um par como `jornada de` pode chamar atenção mesmo com menos ocorrências, caso
seus componentes apareçam juntos em proporção muito maior do que esperaríamos a
partir de suas frequências individuais.

A **informação mútua pontual** (*Pointwise Mutual Information*, PMI) formaliza
essa comparação:

$$
PMI(a,b)=\log_2\left(\frac{P(a,b)}{P_L(a)P_R(b)}\right)
=\log_2\left(\frac{c(a,b)\,N_b}{c_L(a)c_R(b)}\right).
$$

| Símbolo | Significado |
|---|---|
| $c(a,b)$ | número de ocorrências do bigrama $(a,b)$ |
| $N_b$ | número total de bigramas considerados |
| $c_L(a)$ | vezes em que $a$ aparece como primeiro elemento de um bigrama |
| $c_R(b)$ | vezes em que $b$ aparece como segundo elemento de um bigrama |

A ideia central está no quociente dentro do logaritmo. O numerador representa a
**coocorrência observada**; o denominador representa a coocorrência que
esperaríamos se as posições de $a$ e $b$ fossem independentes, dadas suas
frequências marginais.

Isso fornece uma interpretação particularmente simples:

- **PMI = 0**: o par ocorre aproximadamente na frequência esperada pelas marginais;
- **PMI > 0**: as palavras aparecem juntas **mais** do que o esperado;
- **PMI < 0**: aparecem juntas **menos** do que o esperado;
- como usamos $\log_2$, **PMI = 1** corresponde a uma coocorrência duas vezes a
  esperada; **PMI = 2**, quatro vezes; **PMI = 3**, oito vezes.

Portanto, **frequência e PMI respondem a perguntas diferentes**. Frequência
pergunta *“quais pares aparecem mais vezes?”*; PMI pergunta *“quais pares são
especialmente associados, levando em conta quão frequentes são seus componentes?”*

Há uma cautela importante. Um bigrama observado uma única vez pode receber PMI
muito alto se suas duas palavras também forem raras. O valor seria matematicamente
correto, mas pouco robusto para interpretação. Por isso, abaixo calcularemos PMI
somente para bigramas com pelo menos três ocorrências.

Finalmente, PMI é uma medida de **associação distribucional**, não de importância
histórica, significado ou causalidade. Depois de localizar um bigrama interessante,
devemos voltar aos documentos e às concordâncias para verificar em que contextos
ele efetivamente ocorre.

In [ ]:
bigramas = Counter()
marginal_esquerda = Counter()
marginal_direita = Counter()

for tokens_documento in dados["tokens"]:
    pares_documento = list(zip(tokens_documento, tokens_documento[1:]))
    bigramas.update(pares_documento)
    marginal_esquerda.update(a for a, _ in pares_documento)
    marginal_direita.update(b for _, b in pares_documento)

total_bigramas = sum(bigramas.values())
frequencia_minima = 3
linhas_pmi = []
for (a, b), frequencia in bigramas.items():
    if frequencia >= frequencia_minima:
        pmi = math.log2(
            frequencia * total_bigramas
            / (marginal_esquerda[a] * marginal_direita[b])
        )
        linhas_pmi.append((f"{a} {b}", frequencia, pmi))

tabela_colocacoes = pd.DataFrame(
    sorted(linhas_pmi, key=lambda linha: linha[2], reverse=True),
    columns=["bigrama", "frequencia", "pmi"],
)
tabela_colocacoes.head(10)

N-gramas e PMI observam relações locais entre palavras. Agora mudaremos a
escala: em vez de pares de tokens, compararemos o repertório lexical de documentos
inteiros.

## Vocabulário e diversidade lexical

Uma primeira distinção é entre **tokens** e **formas** (*types*).

Considere a sequência:

`trabalho salário trabalho jornada`

Ela contém quatro **tokens**, pois há quatro ocorrências de palavras, mas apenas
três **formas distintas**: `trabalho`, `salário` e `jornada`.

Se $N_d$ é o número total de tokens do documento $d$ e $V_d$ o número de formas
distintas, a **razão forma–token** (*Type-Token Ratio*, TTR) é

$$
TTR(d)=\frac{V_d}{N_d}.
$$

A TTR varia entre 0 e 1. Quanto mais próximo de 1, maior é a proporção de formas
diferentes no trecho analisado. Por exemplo:

- `trabalho salário oficina jornada` tem $TTR=4/4=1$;
- `trabalho trabalho trabalho jornada` tem $TTR=2/4=0{,}5$.

É tentador interpretar a TTR simplesmente como “riqueza de vocabulário”, mas há
um problema fundamental: **ela depende fortemente do tamanho do texto**.

Quando começamos a ler um documento, muitas palavras ainda não apareceram e é
fácil encontrar novas formas. À medida que o texto cresce, palavras já observadas
tendem a reaparecer. O denominador $N_d$ continua aumentando a cada token, enquanto
o número de formas distintas $V_d$ cresce mais lentamente. Assim, mesmo dois
textos produzidos por processos lexicais semelhantes podem apresentar TTRs
diferentes apenas porque um deles é maior.

Por isso, comparar diretamente a TTR de uma carta curta com a de um editorial
muito longo pode levar a uma conclusão enganosa: uma TTR menor no editorial **não
implica, por si só, vocabulário menos diverso**.

### Controlando o tamanho

Uma solução didática simples é comparar trechos com o mesmo número de tokens.
Escolhemos um tamanho $m$ e calculamos

$$
TTR_m(d)=
\frac{\left|\{t_1,\ldots,t_m\}\right|}{m},
\qquad N_d\geq m.
$$

Aqui adotaremos como $m$ o tamanho do menor documento. Dessa forma, todos os 24
documentos entram na comparação e todos contribuem com exatamente o mesmo número
de tokens. O gráfico seguinte permite observar lado a lado:

1. **TTR bruta**, calculada sobre o documento inteiro;
2. **TTR padronizada**, calculada sobre os primeiros $m$ tokens.

Se a relação entre tamanho e TTR enfraquecer depois da padronização, teremos uma
evidência concreta de que parte da diferença observada na TTR bruta era produzida
pelo próprio tamanho dos documentos.

A padronização, contudo, não torna a TTR uma medida perfeita. Usar os primeiros
$m$ tokens torna o resultado sensível à posição escolhida: introduções podem ter
vocabulário diferente do restante do texto. Em uma pesquisa real, poderíamos
comparar várias janelas ou amostras de tamanho fixo e registrar explicitamente o
protocolo utilizado.

Portanto, nesta aula, a TTR deve ser entendida como uma **medida simples da
proporção entre formas e tokens sob um determinado procedimento de amostragem**,
e não como uma medida absoluta de “riqueza”, qualidade ou complexidade lexical.

In [ ]:
tamanho_padrao = int(dados["tokens"].map(len).min())

def ttr_padronizada(tokens, tamanho):
    if len(tokens) < tamanho:
        return pd.NA
    segmento = tokens[:tamanho]
    return len(set(segmento)) / tamanho

diversidade = pd.DataFrame({
    "id_documento": dados["id_documento"],
    "tokens": dados["tokens"].map(len),
    "formas": dados["tokens"].map(lambda tokens: len(set(tokens))),
    "ttr": dados["tokens"].map(lambda tokens: len(set(tokens)) / len(tokens)),
    f"ttr_{tamanho_padrao}": dados["tokens"].map(
        lambda tokens: ttr_padronizada(tokens, tamanho_padrao)
    ),
})
print("Tamanho comum adotado:", tamanho_padrao, "tokens")

display(ttr_duplo(
    diversidade["tokens"].tolist(),
    diversidade["ttr"].tolist(),
    diversidade[f"ttr_{tamanho_padrao}"].tolist(),
    tamanho_padrao,
))
diversidade.head()

A diversidade lexical completa um percurso que começou na representação do
texto, passou pelas contagens e retornou ao contexto. A atividade reúne essas
camadas para selecionar resultados que poderão ser comunicados visualmente sem
perder as regras que os produziram.

## U04-A03 — Atividade — exploração textual

Documente:

1. a regra de tokenização adotada;
2. as palavras de conteúdo mais frequentes e o denominador usado;
3. concordâncias de uma palavra cuja interpretação não seja evidente apenas pela frequência;
4. dois bigramas, comparando frequência e PMI;
5. dois documentos de tamanhos diferentes, comparando TTR bruta e padronizada;
6. uma limitação do corpus sintético.

Separe, ao final, **descrição calculada** de **hipótese interpretativa**. Retorne
a trechos e indique quais resultados seguirão para o Notebook 03.
